# Phase 38 — RoBERTa-large + Focal Loss (train+valid)

Fine-tuning του `roberta-large` με Focal Loss σε train+valid.

**Γιατί RoBERTa-large και όχι base:**
- roberta-base έβγαλε 0.72959 — χειρότερο από BERT-base
- roberta-large έχει 355M params vs 125M του base
- Το large version έχει πολύ καλύτερες representations
- Με Focal Loss αναμένουμε σημαντική βελτίωση

**Best so far:** 0.81882

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import random
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
train = pd.read_csv('/train.csv')
valid = pd.read_csv('/valid.csv')
test  = pd.read_csv('/test.csv')

train_full = pd.concat([train, valid], ignore_index=True)

def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

texts_full = make_text(train_full)
texts_test = make_text(test)

le_hazard  = LabelEncoder()
le_product = LabelEncoder()
le_hazard.fit(train_full['hazard-category'])
le_product.fit(train_full['product-category'])

y_full_hazard  = le_hazard.transform(train_full['hazard-category'])
y_full_product = le_product.transform(train_full['product-category'])

n_hazard  = len(le_hazard.classes_)
n_product = len(le_product.classes_)
print(f'Train+Valid: {len(train_full)} | Test: {len(texts_test)}')
print(f'Hazard classes: {n_hazard} | Product classes: {n_product}')

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
    def forward(self, logits, targets):
        ce    = F.cross_entropy(logits, targets, reduction='none')
        pt    = torch.exp(-ce)
        focal = (1 - pt) ** self.gamma * ce
        return focal.mean()

criterion = FocalLoss(gamma=2.0)
print('Focal Loss initialized! (gamma=2.0)')

In [ ]:
MODEL_NAME   = 'roberta-large'
BATCH_SIZE   = 16    # Μικρότερο batch λόγω μεγέθους μοντέλου
MAX_LENGTH   = 128
LR           = 1e-5  # Μικρότερο LR για large μοντέλο
N_EPOCHS     = 20
WARMUP_RATIO = 0.1

print(f'Φόρτωση tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded!')

dummy_labels = np.zeros(len(texts_test), dtype=int)

In [ ]:
class FoodHazardDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in loader:
        labels = batch.pop('label').to(device)
        batch  = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        loss = criterion(model(**batch).logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def get_probabilities(model, loader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            batch.pop('label', None)
            batch = {k: v.to(device) for k, v in batch.items()}
            probs = torch.softmax(model(**batch).logits, dim=1).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)

In [ ]:
full_loader_hazard  = DataLoader(FoodHazardDataset(texts_full, y_full_hazard,  tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
full_loader_product = DataLoader(FoodHazardDataset(texts_full, y_full_product, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
test_loader_hazard  = DataLoader(FoodHazardDataset(texts_test, dummy_labels,   tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
test_loader_product = DataLoader(FoodHazardDataset(texts_test, dummy_labels,   tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(full_loader_hazard)}')
print(f'Batch size: {BATCH_SIZE} | Epochs: {N_EPOCHS} | LR: {LR}')

## Εκπαίδευση RoBERTa-large — Hazard

In [ ]:
print('=== ΕΚΠΑΙΔΕΥΣΗ RoBERTa-large + Focal Loss — HAZARD ===')
print(f'Δείγματα: {len(texts_full)} | Epochs: {N_EPOCHS} | LR: {LR}\n')

roberta_hazard = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=n_hazard
).to(device)

opt_h        = AdamW(roberta_hazard.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(full_loader_hazard) * N_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
sch_h        = get_linear_schedule_with_warmup(opt_h, warmup_steps, total_steps)
print(f'Total steps: {total_steps} | Warmup: {warmup_steps}\n')

for epoch in range(N_EPOCHS):
    loss = train_epoch(roberta_hazard, full_loader_hazard, opt_h, sch_h)
    print(f'Epoch {epoch+1}/{N_EPOCHS} | Focal Loss: {loss:.4f}')

print('\nHazard εκπαίδευση ολοκληρώθηκε')

## Εκπαίδευση RoBERTa-large — Product

In [ ]:
print('=== ΕΚΠΑΙΔΕΥΣΗ RoBERTa-large + Focal Loss — PRODUCT ===')
print(f'Δείγματα: {len(texts_full)} | Epochs: {N_EPOCHS} | LR: {LR}\n')

roberta_product = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=n_product
).to(device)

opt_p        = AdamW(roberta_product.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(full_loader_product) * N_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
sch_p        = get_linear_schedule_with_warmup(opt_p, warmup_steps, total_steps)
print(f'Total steps: {total_steps} | Warmup: {warmup_steps}\n')

for epoch in range(N_EPOCHS):
    loss = train_epoch(roberta_product, full_loader_product, opt_p, sch_p)
    print(f'Epoch {epoch+1}/{N_EPOCHS} | Focal Loss: {loss:.4f}')

print('\nProduct εκπαίδευση ολοκληρώθηκε')

## Αποθήκευση & Submission

In [ ]:
# Test probabilities
roberta_test_hazard_probs  = get_probabilities(roberta_hazard,  test_loader_hazard)
roberta_test_product_probs = get_probabilities(roberta_product, test_loader_product)

np.save('robertalarge_focal_test_hazard_probs.npy',  roberta_test_hazard_probs)
np.save('robertalarge_focal_test_product_probs.npy', roberta_test_product_probs)
print('Αποθηκεύτηκαν τα .npy για ensemble')

# Submission RoBERTa-large μόνο
test_hazard  = le_hazard.inverse_transform(roberta_test_hazard_probs.argmax(axis=1))
test_product = le_product.inverse_transform(roberta_test_product_probs.argmax(axis=1))

pd.DataFrame({
    'id': test['id'],
    'hazard-category': test_hazard,
    'product-category': test_product
}).to_csv('submission_robertalarge_focal.csv', index=False)

print('Αποθηκεύτηκε: submission_robertalarge_focal.csv')
print('\n=== ΣΥΓΚΡΙΣΗ ===')
print('RoBERTa-base Focal:  0.72959')
print('BERT-base Focal:     0.80399')
print('Best ensemble:       0.81882')